# Google Colab edition

Run the setup cell first, then work from top to bottom. Keep the course ZIP in My Drive. Use a fresh Colab runtime when switching dependency profiles. **Select a GPU runtime before running this notebook.** 

Original numbered course notebook, adapted for Colab. Run in numerical order: later lessons load models trained by earlier lessons. Use the same Drive result folder across sessions. W&B logs stay offline. Requires your own ImageNet CLS-LOC data under data/ILSVRC; it is not included or replaced with synthetic data.

Source: https://github.com/fastai/course22p2/blob/df9323235bc395b5c2f58a3d08b83761947b9b93/nbs/31_imgnet_latents-widish.ipynb

In [ ]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '04_fastai_part2', 'part2', '31_imgnet_latents-widish.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)

## Full exercise evidence

- Run all exercise cells top to bottom in hosted Colab and preserve their outputs, errors and execution counts.
- Keep QUICK_RUN=False; shortened checks or synthetic replacements for real datasets do not count as full original exercises.
- Preserve plots, models, metrics and run configurations in Drive; a final-cell marker alone is insufficient.
- Retain original datasets, model definitions, training schedules and instructional comparisons.
- Run prerequisite checkpoint-producing notebooks first; do not replace missing weights with untrained models.
- Supply the full ILSVRC image and synset inputs permitted by their terms. Tiny ImageNet is a different experiment.

- Before the original40-epoch1,000-class classifier, require1,281,167 training and50,000 validation originals with exact synset mapping/order; validate every finite float32(4,32,32) latent against current source SHA256 and exact VAE/preprocessing identity. Repair only missing/corrupt/current-source-invalid rows; partial caches cannot pass.

This cell describes evidence to collect; it does not certify completion. GPU batch caps preserve dataset coverage and epoch counts, but are documented configuration changes.

In [ ]:
MAX_BATCH_SIZE = 8  #@param {type:"integer"}
# Increase on a larger GPU to match the original batch sizes.
assert MAX_BATCH_SIZE > 0

# Full Imagenet

In [ ]:
import os
# Colab provides the visible GPU; keep its device selection.
os.environ['OMP_NUM_THREADS']='1'

In [ ]:
import pickle,gzip

from glob import glob
from torcheval.metrics import MulticlassAccuracy

from miniai.imports import *

In [ ]:
from fastprogress import progress_bar
from diffusers import AutoencoderKL

In [ ]:
torch.set_printoptions(precision=5, linewidth=140, sci_mode=False)
torch.manual_seed(1)
mpl.rcParams['figure.dpi'] = 70

set_seed(42)
if fc.defaults.cpus>2: fc.defaults.cpus=2

In [ ]:
path_data = Path('data')/'ILSVRC'
path = path_data/'Data'/'CLS-LOC'

dest = path_data/'latents'
dest.mkdir(parents=True, exist_ok=True)

In [ ]:
vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-ema").cuda().requires_grad_(False)

In [ ]:
class ImagesDS:
    def __init__(self,path,spec):
        # Re-scan current originals; a stale files.zpkl cannot certify input completeness.
        self.files = glob(str(path/spec),recursive=True)
    def __len__(self): return len(self.files)
    def __getitem__(self,i):
        f=self.files[i]
        im=read_image(f,mode=ImageReadMode.RGB)/255
        im=TF.resize(TF.center_crop(im,min(im.shape[1:])),256)
        return im,f

In [ ]:
"""Inlined into notebooks30/31; functions only, with no downloads or training on import."""
import csv, gzip, hashlib, json, os, shutil, sqlite3, tarfile, time
from pathlib import Path
import numpy as np

def cache_sha_file(filename):
    value = hashlib.sha256()
    with open(filename, 'rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            value.update(block)
    return value.hexdigest()

def cache_json_sha(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',',':'), default=str).encode()).hexdigest()

def cache_atomic_json(filename, value):
    filename = Path(filename); filename.parent.mkdir(parents=True, exist_ok=True)
    temporary = filename.with_name(filename.name+'.writing')
    with temporary.open('w', encoding='utf-8') as stream:
        json.dump(value, stream, sort_keys=True, indent=2); stream.flush(); os.fsync(stream.fileno())
    os.replace(temporary, filename)

def latent_npy_disk_plan(cache_root, rows):
    """Account for NPY headers, filesystem blocks and available file entries."""
    import io
    cache_root = Path(cache_root)
    sample = io.BytesIO(); np.save(sample,np.empty((4,32,32),np.float32),allow_pickle=False)
    logical_row_bytes = sample.tell()
    fs = os.statvfs(cache_root) if hasattr(os,'statvfs') else None
    block = (fs.f_frsize or fs.f_bsize) if fs else 512
    allocated_row_bytes = ((logical_row_bytes+block-1)//block)*block
    existing = list(cache_root.rglob('*.npy'))
    allocated = sum(min(getattr(p.stat(),'st_blocks',0)*512 or p.stat().st_size,allocated_row_bytes) for p in existing)
    required_free = max(rows*allocated_row_bytes-allocated,0)+1024**3
    if shutil.disk_usage(cache_root).free < required_free:
        raise OSError(f'Full original float32 NPY cache needs {required_free:,} additional runtime bytes, including file-block overhead and manifest reserve.')
    if fs and fs.f_files and fs.f_favail < max(rows-len(existing),0)+2100:
        raise OSError('Runtime filesystem has insufficient free file entries for the complete original NPY cache.')
    return {'npy_bytes_per_row':logical_row_bytes,'filesystem_block_bytes':block,
            'allocated_bytes_per_row':allocated_row_bytes,'full_allocated_bytes':rows*allocated_row_bytes}

def full_lsun_source(path, url, evidence, expected_images=303125):
    """Verify every original tar member; repair missing/corrupt JPEGs atomically."""
    import urllib.request
    path = Path(path); evidence = Path(evidence); evidence.mkdir(parents=True, exist_ok=True)
    archive = path.parent/'bedroom.tgz'
    if not archive.is_file():
        request = urllib.request.Request(url, method='HEAD')
        with urllib.request.urlopen(request, timeout=60) as response:
            advertised_bytes = int(response.headers.get('Content-Length',0))
        free = shutil.disk_usage(path.parent).free
        if advertised_bytes and free < advertised_bytes + 1024**3:
            raise OSError('Insufficient runtime disk for original LSUN archive; data are not stored in Drive.')
        partial = archive.with_name(archive.name+'.downloading')
        # No partial HTTP download is treated as a complete archive.
        with urllib.request.urlopen(url, timeout=120) as response, partial.open('wb') as output:
            shutil.copyfileobj(response, output, 1024*1024); output.flush(); os.fsync(output.fileno())
        if advertised_bytes and partial.stat().st_size != advertised_bytes:
            raise IOError('LSUN archive download is truncated.')
        os.replace(partial, archive)
    source_hashes = {}; aggregate = hashlib.sha256(); extracted_bytes = 0
    source_record = evidence/'original_lsun_source.jsonl.gz'
    temporary_record = source_record.with_name(source_record.name+'.writing')
    try:
        with tarfile.open(archive, 'r:gz') as source, gzip.open(temporary_record, 'wt', encoding='utf-8') as manifest:
            members = [m for m in source.getmembers() if m.isfile() and m.name.lower().endswith('.jpg')]
            if len(members) != expected_images:
                raise ValueError(f'Original LSUN archive must contain {expected_images} JPEGs; found {len(members)}.')
            expected_targets = set()
            missing_bytes = 0
            for member in members:
                parts = Path(member.name).parts
                if not parts or parts[0] != 'bedroom' or '..' in parts or Path(member.name).is_absolute():
                    raise ValueError('Unsafe or unexpected original LSUN archive path.')
                target = (path.parent/member.name).resolve()
                if not target.is_relative_to(path.resolve()) or target in expected_targets:
                    raise ValueError('Unsafe or duplicate original LSUN JPEG path.')
                expected_targets.add(target); extracted_bytes += member.size
                if not target.is_file() or target.stat().st_size != member.size:
                    missing_bytes += member.size
            if shutil.disk_usage(path.parent).free < missing_bytes + 1024**3:
                raise OSError('Insufficient runtime disk for extracted LSUN images; use a larger runtime disk.')
            for member in members:
                target = (path.parent/member.name).resolve(); target.parent.mkdir(parents=True, exist_ok=True)
                # Hash the authoritative JPEG bytes inside the course's exact archive.
                with source.extractfile(member) as input_stream:
                    original_bytes = input_stream.read()
                if len(original_bytes) != member.size: raise IOError('Truncated JPEG inside original LSUN archive.')
                source_sha = hashlib.sha256(original_bytes).hexdigest()
                if not target.is_file() or target.stat().st_size != member.size or cache_sha_file(target) != source_sha:
                    temporary = target.with_name(target.name+'.restoring')
                    with temporary.open('wb') as output:
                        output.write(original_bytes); output.flush(); os.fsync(output.fileno())
                    if temporary.stat().st_size != member.size or cache_sha_file(temporary) != source_sha:
                        raise IOError('LSUN JPEG restoration failed: '+member.name)
                    os.replace(temporary, target)
                relative = target.relative_to(path.resolve()).as_posix()
                source_hashes[relative] = source_sha
                line = json.dumps({'relative_path':relative,'bytes':member.size,'sha256':source_sha}, separators=(',',':'))
                aggregate.update((line+'\n').encode()); manifest.write(line+'\n')
            actual = {p.resolve() for p in path.rglob('*.jpg')}
            if actual != expected_targets:
                raise ValueError('LSUN directory differs from the exact archive JPEG inventory; no extra/omitted images are allowed.')
        os.replace(temporary_record, source_record)
        summary = {'original_url':url,'archive_bytes':archive.stat().st_size,'archive_sha256':cache_sha_file(archive),
                   'extracted_jpeg_bytes':extracted_bytes,'expected_images':expected_images,
                   'verified_images':len(source_hashes),'source_manifest_sha256':aggregate.hexdigest(),
                   'data_location':'runtime-local','course_completion_claimed':False}
        cache_atomic_json(evidence/'original_lsun_source_status.json', summary)
        print('Verified exact LSUN source:',summary)
        return source_hashes
    except BaseException as error:
        cache_atomic_json(evidence/'original_lsun_source_status.json', {'source_complete':False,'error':str(error),'course_completion_claimed':False})
        raise

def full_imagenet_source(path, path_data, files, expected_train=1281167, expected_val=50000, expected_classes=1000):
    """Require exact full splits and resolve official labels, including flat val JPEGs."""
    import xml.etree.ElementTree as ET
    path = Path(path); path_data = Path(path_data)
    id_file = path_data/'imagenet_lsvrc_2015_synsets.txt'; words_file = path_data/'words.txt'
    if not id_file.is_file() or not words_file.is_file():
        raise FileNotFoundError('Supply original ImageNet synset-order and words files under data/ILSVRC; no Tiny ImageNet replacement is used.')
    ids = id_file.read_text().splitlines(); id_set = set(ids)
    if len(ids) != expected_classes or len(id_set) != expected_classes or any(not s.startswith('n') or not s[1:].isdigit() for s in ids):
        raise ValueError('ImageNet synset-order file must contain exactly1000 distinct original synsets.')
    words = dict(line.split('\t',1) for line in words_file.read_text().splitlines() if '\t' in line)
    if not id_set.issubset(words): raise ValueError('ImageNet words file does not cover all required synsets.')
    val_csv = next((p for p in [path_data/'LOC_val_solution.csv',path_data.parent/'LOC_val_solution.csv'] if p.is_file()), None)
    csv_labels = {}
    if val_csv:
        with val_csv.open(newline='') as stream:
            for row in csv.DictReader(stream):
                labels = {word for word in row['PredictionString'].split() if word in id_set}
                if len(labels) != 1: raise ValueError('Official validation row has ambiguous classification synset: '+row['ImageId'])
                csv_labels[row['ImageId']] = next(iter(labels))
    targets = []; counts = {'train':0,'val':0,'test':0}; class_counts = {key:0 for key in ids}; val_class_counts = {key:0 for key in ids}; label_inputs = [id_file,words_file]
    if val_csv: label_inputs.append(val_csv)
    annotation_hash = hashlib.sha256(); seen = set()
    for filename in files:
        filename = Path(filename); relative = filename.resolve().relative_to(path.resolve())
        split = relative.parts[0]
        if split not in counts: raise ValueError('Unexpected CLS-LOC split: '+str(relative))
        counts[split] += 1
        if split == 'train':
            label = relative.parts[1]
            if label not in id_set: raise ValueError('Unknown training synset: '+label)
            class_counts[label] += 1
        elif split == 'val':
            label = filename.parent.name if filename.parent.name in id_set else csv_labels.get(filename.stem)
            if label is None:
                annotation = path_data/'Annotations'/'CLS-LOC'/'val'/(filename.stem+'.xml')
                if not annotation.is_file(): raise FileNotFoundError('Flat ImageNet validation JPEGs require their official val XML or LOC_val_solution.csv: '+filename.name)
                labels = {node.text for node in ET.parse(annotation).findall('./object/name')}
                if len(labels) != 1 or not labels.issubset(id_set): raise ValueError('Ambiguous original validation annotation: '+annotation.name)
                label = next(iter(labels)); annotation_hash.update((annotation.name+cache_sha_file(annotation)).encode())
            val_class_counts[label] += 1
        else: label = ''
        target = Path(split)/(label or 'unlabelled')/(filename.stem+'.npy')
        if target.as_posix() in seen: raise ValueError('Duplicate latent target: '+str(target))
        seen.add(target.as_posix()); targets.append(target.as_posix())
    if counts['train'] != expected_train or counts['val'] != expected_val or not all(class_counts.values()):
        raise ValueError(f'Full ImageNet requires train={expected_train}, val={expected_val}, all{expected_classes}classes; observed {counts}.')
    if expected_val % expected_classes or any(count != expected_val//expected_classes for count in val_class_counts.values()):
        raise ValueError('Original ImageNet validation must include exactly50 images for each of the1000 synsets.')
    if counts['test'] not in [0,100000]: raise ValueError('If supplied, the optional unlabelled test split must be complete.')
    labels_fingerprint = cache_json_sha({'files':{p.name:cache_sha_file(p) for p in label_inputs},'xml_labels_sha256':annotation_hash.hexdigest(),'synset_order':ids})
    return targets, {'counts':counts,'class_counts':class_counts,'validation_class_counts':val_class_counts,'synset_order_sha256':cache_sha_file(id_file),'label_sources_sha256':labels_fingerprint}

def latent_encoder_settings(encoder, preprocessing):
    import torch, torchvision
    weights = hashlib.sha256()
    for name, tensor in sorted(encoder.state_dict().items()):
        array = tensor.detach().cpu().contiguous().numpy()
        weights.update(name.encode()); weights.update(str(array.shape).encode()); weights.update(str(array.dtype).encode()); weights.update(array.tobytes())
    return {'model_id':'stabilityai/sd-vae-ft-ema','weights_sha256':weights.hexdigest(),
            'config':dict(encoder.config),'preprocessing':preprocessing,
            'latent_statistic':'latent_dist.mean','shape':[4,32,32],'dtype':'float32',
            'torch':torch.__version__,'torchvision':torchvision.__version__}

def verified_full_latents(files, source_root, get_image, encoder, cache_base, evidence, settings,
                          mode='npy', targets=None, batch_size=4, source_proof=None, snapshot_seconds=300):
    """Every row must have current source and output hashes; interrupted rows resume."""
    import torch
    files = [Path(p) for p in files]; source_root = Path(source_root).resolve()
    relatives = [p.resolve().relative_to(source_root).as_posix() for p in files]
    if len(set(relatives)) != len(files) or not files: raise ValueError('Source inventory is empty or duplicated.')
    if mode not in ['npy','memmap']: raise ValueError(mode)
    if targets is None: targets = [str(i) for i in range(len(files))]
    if len(targets) != len(files) or len(set(targets)) != len(files): raise ValueError('Latent row manifest is not one-to-one.')
    order = hashlib.sha256()
    for relative,target in zip(relatives,targets): order.update((relative+'\t'+target+'\n').encode())
    identity = {'settings':settings,'source_order_sha256':order.hexdigest(),'rows':len(files),'mode':mode}
    identity_sha = cache_json_sha(identity)
    cache_root = Path(cache_base)/('verified-'+identity_sha[:24]); cache_root.mkdir(parents=True, exist_ok=True)
    evidence = Path(evidence); evidence.mkdir(parents=True, exist_ok=True)
    snapshot = evidence/(identity_sha[:24]+'-manifest.sqlite3.gz')
    db_file = cache_root/'row-manifest.sqlite3'
    if not db_file.exists() and snapshot.exists():
        temporary = db_file.with_name(db_file.name+'.restoring')
        with gzip.open(snapshot,'rb') as source, temporary.open('wb') as output: shutil.copyfileobj(source,output)
        os.replace(temporary,db_file)
    conn = sqlite3.connect(db_file)
    conn.execute('PRAGMA synchronous=FULL')
    if conn.execute('PRAGMA integrity_check').fetchone()[0] != 'ok': raise IOError('Corrupt latent row manifest; never treat it as completion.')
    conn.execute('CREATE TABLE IF NOT EXISTS metadata (key TEXT PRIMARY KEY,value TEXT)')
    conn.execute('CREATE TABLE IF NOT EXISTS rows (seq INTEGER PRIMARY KEY,relative TEXT,source_sha TEXT,latent_sha TEXT,complete INTEGER)')
    old = conn.execute("SELECT value FROM metadata WHERE key='identity'").fetchone()
    if old and old[0] != identity_sha: raise ValueError('Latent manifest encoding/source-order identity does not match.')
    conn.execute("INSERT OR REPLACE INTO metadata VALUES ('identity',?)",(identity_sha,)); conn.commit()
    output_file = cache_root/'data.npmm'; array = None
    shape = (len(files),4,32,32); total_bytes = int(np.prod(shape,dtype=np.int64))*np.dtype(np.float32).itemsize
    if mode == 'memmap':
        if output_file.exists() and output_file.stat().st_size != total_bytes:
            os.replace(output_file,output_file.with_name('invalid-size-'+str(time.time_ns())+'.npmm'))
            conn.execute('UPDATE rows SET complete=0');conn.commit()
        if not output_file.exists():
            if shutil.disk_usage(cache_root).free < total_bytes+1024**3: raise OSError('Insufficient runtime disk for full float32 latent memmap.')
            conn.execute('UPDATE rows SET complete=0');conn.commit()
            allocation = output_file.with_name(output_file.name+'.allocating')
            fresh = np.memmap(allocation,np.float32,mode='w+',shape=shape);fresh.flush();del fresh
            os.replace(allocation,output_file)
        array = np.memmap(output_file,np.float32,mode='r+',shape=shape)
    else: latent_npy_disk_plan(cache_root,len(files))
    started = time.monotonic(); last_snapshot = started; verified = 0; regenerated = 0; aggregate = hashlib.sha256(); ready = False
    status_file = evidence/(identity_sha[:24]+'-status.json')
    def checkpoint(error=None):
        nonlocal last_snapshot
        conn.commit(); copy_file = cache_root/'manifest-snapshot.sqlite3'
        copy = sqlite3.connect(copy_file)
        try: conn.backup(copy)
        finally: copy.close()
        temporary = snapshot.with_name(snapshot.name+'.writing')
        with copy_file.open('rb') as source, gzip.open(temporary,'wb',compresslevel=1) as output: shutil.copyfileobj(source,output,1024*1024)
        os.replace(temporary,snapshot);copy_file.unlink()
        cache_atomic_json(status_file,{'identity':identity,'identity_sha256':identity_sha,'expected_rows':len(files),
            'verified_current_source_rows':verified,'regenerated_this_attempt':regenerated,'cache_complete':ready,
            'source_content_manifest_sha256':aggregate.hexdigest(),'duration_seconds':time.monotonic()-started,
            'runtime_cache':str(cache_root),'latent_payload_bytes':total_bytes,'error':error,
            'data_location':'runtime-local; only compact manifest/status persisted','course_completion_claimed':False})
        last_snapshot = time.monotonic()
    def valid_latent(value):
        return value.shape == (4,32,32) and value.dtype == np.float32 and bool(np.isfinite(value).all())
    def read_latent(index):
        return np.asarray(array[index]) if mode == 'memmap' else np.load(cache_root/targets[index],allow_pickle=False)
    def process(todo):
        nonlocal regenerated
        if not todo: return
        batch = torch.stack([get_image(index) for index,sha in todo])
        device = next(encoder.parameters()).device
        with torch.inference_mode(): values = encoder.encode(batch.to(device)).latent_dist.mean.detach().cpu().numpy()
        if values.shape != (len(todo),4,32,32) or values.dtype != np.float32 or not np.isfinite(values).all():
            raise ValueError('Encoder did not produce complete finite original float32 mean latents.')
        rows = []
        for (index,sha),value in zip(todo,values):
            if cache_sha_file(files[index]) != sha: raise IOError('Source image changed during latent encoding: '+relatives[index])
            if mode == 'memmap': array[index] = value
            else:
                output = cache_root/targets[index];output.parent.mkdir(parents=True,exist_ok=True)
                if not output.resolve().is_relative_to(cache_root.resolve()): raise ValueError('Unsafe latent output path.')
                temporary = output.with_name(output.name+'.writing')
                with temporary.open('wb') as stream:
                    np.save(stream,value,allow_pickle=False);stream.flush();os.fsync(stream.fileno())
                os.replace(temporary,output)
            rows.append((index,relatives[index],sha,hashlib.sha256(value.tobytes()).hexdigest(),1))
        if mode == 'memmap':
            array.flush()
            with output_file.open('r+b') as stream: os.fsync(stream.fileno())
        conn.executemany('INSERT OR REPLACE INTO rows VALUES (?,?,?,?,?)',rows);conn.commit();regenerated += len(rows)
    try:
        checkpoint(); todo = []
        for index,filename in enumerate(files):
            # Re-read current source hashes on every attempt; metadata-only cache hits are insufficient.
            sha = cache_sha_file(filename)
            if source_proof is not None and source_proof.get(relatives[index]) != sha:
                raise ValueError('Current LSUN image differs from the verified original archive: '+relatives[index])
            aggregate.update((relatives[index]+'\t'+sha+'\t'+targets[index]+'\n').encode())
            row = conn.execute('SELECT relative,source_sha,latent_sha,complete FROM rows WHERE seq=?',(index,)).fetchone()
            valid = False
            if row and row[0] == relatives[index] and row[1] == sha and row[3]:
                try:
                    value = read_latent(index)
                    valid = valid_latent(value) and hashlib.sha256(value.tobytes()).hexdigest() == row[2]
                except (OSError,ValueError,EOFError): pass
            if not valid: todo.append((index,sha))
            if len(todo) >= batch_size: process(todo);todo=[]
            verified = index+1
            if time.monotonic()-last_snapshot >= snapshot_seconds:
                process(todo);todo=[];checkpoint()
        process(todo)
        if conn.execute('SELECT COUNT(*) FROM rows WHERE complete=1').fetchone()[0] != len(files):
            raise ValueError('Latent row manifest is incomplete; training is blocked.')
        if mode == 'npy':
            actual = {p.relative_to(cache_root).as_posix() for p in cache_root.rglob('*.npy')}
            if actual != set(targets): raise ValueError('Latent file inventory has extra or missing rows.')
        ready = True;checkpoint()
        print(f'Verified {len(files):,} current-source latent rows; regenerated {regenerated:,}; data remain runtime-local.')
        return {'cache_root':cache_root,'memmap':output_file if mode=='memmap' else None,
                'files':[str(cache_root/target) for target in targets] if mode=='npy' else [],
                'identity_sha256':identity_sha,'source_manifest_sha256':aggregate.hexdigest(),
                'verified_rows':len(files),'regenerated_rows':regenerated,'cache_complete':True}
    except BaseException as error:
        ready = False;checkpoint(str(error));raise
    finally:
        if array is not None: array.flush();del array
        conn.close()

In [ ]:
"""Optional original ImageNet streaming; definitions alone do not access any service."""

def imagenet_original_metadata(path_data, expected_classes=1000):
    path_data = Path(path_data)
    ids_file = path_data/'imagenet_lsvrc_2015_synsets.txt'; words_file = path_data/'words.txt'
    if not ids_file.is_file() or not words_file.is_file():
        raise FileNotFoundError('Supply the original 1000-synset order and words files under data/ILSVRC before using either source route.')
    ids = ids_file.read_text().splitlines()
    if len(ids)!=expected_classes or len(set(ids))!=expected_classes or any(not s.startswith('n') or not s[1:].isdigit() for s in ids):
        raise ValueError('Original synset file must contain exactly 1000 distinct WNIDs.')
    words = dict(line.split('\t',1) for line in words_file.read_text().splitlines() if '\t' in line)
    if not set(ids).issubset(words): raise ValueError('Original words file does not cover all required synsets.')
    return ids,{'synset_order_sha256':cache_sha_file(ids_file),'words_sha256':cache_sha_file(words_file)}

def verified_stream_label_map(features, original_ids, expected_classes=1000):
    """Use actual revision-bound WNID ClassLabels; never infer display-name order."""
    from datasets import ClassLabel, Image
    if features is None or 'image' not in features or not isinstance(features['image'],Image):
        raise ValueError('The original dataset revision must expose the image feature.')
    candidates = [key for key,value in features.items() if isinstance(value,ClassLabel)]
    if len(candidates)!=1: raise ValueError('No unambiguous original classification ClassLabel feature was found.')
    label_key = candidates[0]; names = list(features[label_key].names)
    if len(names)!=expected_classes or len(set(names))!=expected_classes or set(names)!=set(original_ids):
        raise ValueError('Cannot verify numeric label-to-WNID mapping: exact revision features must contain all 1000 original WNIDs. Human display names, sorted-order guesses and alternate datasets are refused.')
    return label_key,names,cache_json_sha({'field':label_key,'index_to_wnid':names,'features':features.to_dict()})

def approved_imagenet_stream_source(path_data):
    """Only called after the learner opts in with an approved account and token."""
    import datasets, huggingface_hub, re
    from datasets import load_dataset,load_dataset_builder,Image
    from huggingface_hub import HfApi
    if os.environ.get('IMAGENET_APPROVED_ACCESS')!='1':
        raise PermissionError('Set IMAGENET_APPROVED_ACCESS=1 only after personally accepting the official dataset access conditions. This notebook does not accept terms or share contact details.')
    token = os.environ.get('HF_TOKEN')
    if not token: raise PermissionError('Provide your own approved HF_TOKEN through Colab Secrets or the environment; do not put tokens in notebook text.')
    repo = 'ILSVRC/imagenet-1k'
    requested = os.environ.get('IMAGENET_HF_REVISION','main')
    revision = HfApi(token=token).dataset_info(repo,revision=requested).sha
    if not isinstance(revision,str) or not re.fullmatch('[0-9a-f]{40}',revision):
        raise ValueError('The official dataset revision could not be pinned to an immutable commit.')
    ids,original_metadata = imagenet_original_metadata(path_data)
    builder = load_dataset_builder(repo,revision=revision,token=token)
    expected = {'train':1281167,'validation':50000}
    if builder.info.splits is None or any(split not in builder.info.splits or builder.info.splits[split].num_examples!=count for split,count in expected.items()):
        raise ValueError('The pinned official dataset metadata does not prove the exact required full train/validation split counts.')
    label_key,label_map,label_sha = verified_stream_label_map(builder.info.features,ids)
    streams = {}
    for split in expected:
        data = load_dataset(repo,revision=revision,split=split,streaming=True,token=token)
        current_key,current_map,current_sha = verified_stream_label_map(data.features,ids)
        if (current_key,current_map,current_sha)!=(label_key,label_map,label_sha):
            raise ValueError('Builder and streaming split label features differ; mapping is not verified.')
        streams[split] = data.cast_column('image',Image(decode=False))
    source = {'repo':repo,'revision':revision,'requested_revision':requested,'original_metadata':original_metadata,
              'label_field':label_key,'index_to_wnid':label_map,'label_mapping_sha256':label_sha,
              'expected_counts':expected,'expected_classes':1000,'raw_storage':'streamed, not retained',
              'datasets':datasets.__version__,'huggingface_hub':huggingface_hub.__version__,
              'official_card':'https://huggingface.co/datasets/ILSVRC/imagenet-1k'}
    # The token is deliberately absent from persisted source/settings/manifest objects.
    return {'streams':streams,'source':source,'original_ids':ids}

def imagenet_stream_image(blob):
    """Identical JPEG decoding and geometric transforms to the original raw route."""
    import torch
    from torchvision.io import decode_image,ImageReadMode
    from torchvision.transforms import functional as VF
    if not isinstance(blob,(bytes,bytearray)) or not blob: raise ValueError('Original JPEG bytes are unavailable; no alternate image source is used.')
    im = decode_image(torch.frombuffer(bytearray(blob),dtype=torch.uint8),mode=ImageReadMode.RGB)/255
    return VF.resize(VF.center_crop(im,min(im.shape[1:])),256)

def imagenet_stream_preview(bundle,batch_size=4):
    import torch
    images=[];keys=[]
    for index,example in enumerate(bundle['streams']['train'].take(batch_size)):
        image = example['image']; images.append(imagenet_stream_image(image.get('bytes')))
        keys.append(image.get('path') or f'preview-original-stream-row-{index}')
    if len(images)!=batch_size: raise ValueError('Original streaming source ended during preview.')
    return torch.stack(images),keys

def verified_streamed_imagenet_latents(bundle,encoder,cache_base,evidence,settings,batch_size=4,snapshot_seconds=300):
    """Stream every original row, recording revision, order, source/label/output hashes."""
    import torch
    source = bundle['source']; expected = source['expected_counts']; label_map = source['index_to_wnid']
    ids = bundle['original_ids']; id_set = set(ids)
    if set(expected)!={'train','validation'} or source['expected_classes']!=len(id_set):
        raise ValueError('Stream source count/class metadata is invalid.')
    if set(label_map)!=id_set or len(label_map)!=len(ids): raise ValueError('Stream mapping is not the verified original mapping.')
    identity = {'source':source,'settings':settings,'mode':'original_hf_stream_float32_npy','expected_rows':sum(expected.values())}
    identity_sha = cache_json_sha(identity)
    cache_root = Path(cache_base)/('verified-stream-'+identity_sha[:24]);cache_root.mkdir(parents=True,exist_ok=True)
    evidence = Path(evidence);evidence.mkdir(parents=True,exist_ok=True)
    disk = latent_npy_disk_plan(cache_root,sum(expected.values()))
    snapshot = evidence/(identity_sha[:24]+'-stream-manifest.sqlite3.gz')
    status_file = evidence/(identity_sha[:24]+'-stream-status.json')
    db_file = cache_root/'stream-row-manifest.sqlite3'
    if not db_file.exists() and snapshot.is_file():
        temporary = db_file.with_name(db_file.name+'.restoring')
        with gzip.open(snapshot,'rb') as input_stream,temporary.open('wb') as output: shutil.copyfileobj(input_stream,output)
        os.replace(temporary,db_file)
    conn = sqlite3.connect(db_file);conn.execute('PRAGMA synchronous=FULL')
    if conn.execute('PRAGMA integrity_check').fetchone()[0]!='ok':
        conn.close();raise IOError('Corrupt streaming source manifest cannot certify a complete cache.')
    conn.execute('CREATE TABLE IF NOT EXISTS metadata (key TEXT PRIMARY KEY,value TEXT)')
    conn.execute('CREATE TABLE IF NOT EXISTS rows (split TEXT,seq INTEGER,image_key TEXT,source_sha TEXT,label_index INTEGER,wnid TEXT,target TEXT,latent_sha TEXT,complete INTEGER,PRIMARY KEY(split,seq))')
    previous = conn.execute("SELECT value FROM metadata WHERE key='identity'").fetchone()
    if previous and previous[0]!=identity_sha: conn.close();raise ValueError('Streaming revision/encoder identity mismatch.')
    conn.execute("INSERT OR REPLACE INTO metadata VALUES ('identity',?)",(identity_sha,));conn.commit()
    started=time.monotonic();last_snapshot=started;counts={split:0 for split in expected}
    classes={split:{key:0 for key in ids} for split in expected};aggregate=hashlib.sha256();regenerated=0;ready=False
    def checkpoint(error=None):
        nonlocal last_snapshot
        conn.commit();copy_file=cache_root/'stream-manifest-snapshot.sqlite3';copy=sqlite3.connect(copy_file)
        try: conn.backup(copy)
        finally: copy.close()
        temporary=snapshot.with_name(snapshot.name+'.writing')
        with copy_file.open('rb') as input_stream,gzip.open(temporary,'wb',compresslevel=1) as output: shutil.copyfileobj(input_stream,output,1024*1024)
        os.replace(temporary,snapshot);copy_file.unlink()
        cache_atomic_json(status_file,{'identity':identity,'identity_sha256':identity_sha,'observed_counts':counts,
            'class_counts':classes,'cache_complete':ready,'regenerated_this_attempt':regenerated,
            'ordered_source_labels_sha256':aggregate.hexdigest(),'duration_seconds':time.monotonic()-started,
            'disk_plan':disk,'runtime_cache':str(cache_root),'error':error,
            'raw_images_retained':False,'latent_precision':'float32','course_completion_claimed':False})
        last_snapshot=time.monotonic()
    def process(todo):
        nonlocal regenerated
        if not todo: return
        device=next(encoder.parameters()).device
        batch=torch.stack([image for row,image in todo])
        with torch.inference_mode(): values=encoder.encode(batch.to(device)).latent_dist.mean.detach().cpu().numpy()
        if values.shape!=(len(todo),4,32,32) or values.dtype!=np.float32 or not np.isfinite(values).all():
            raise ValueError('The original VAE must return finite float32 mean latents with shape4x32x32.')
        rows=[]
        for (row,image),value in zip(todo,values):
            split,index,key,sha,label,wnid,target=row
            output=cache_root/target;output.parent.mkdir(parents=True,exist_ok=True)
            if not output.resolve().is_relative_to(cache_root.resolve()): raise ValueError('Unsafe streaming latent path.')
            temporary=output.with_name(output.name+'.writing')
            with temporary.open('wb') as stream: np.save(stream,value,allow_pickle=False);stream.flush();os.fsync(stream.fileno())
            os.replace(temporary,output)
            rows.append((*row,hashlib.sha256(value.tobytes()).hexdigest(),1))
        conn.executemany('INSERT OR REPLACE INTO rows VALUES (?,?,?,?,?,?,?,?,?)',rows);conn.commit();regenerated+=len(rows)
    try:
        checkpoint();todo=[]
        for split in ['train','validation']:
            output_split='val' if split=='validation' else 'train'
            for index,example in enumerate(bundle['streams'][split]):
                if index>=expected[split]: raise ValueError('Official streaming split has more rows than required: '+split)
                label=example[source['label_field']]
                if isinstance(label,bool) or not isinstance(label,(int,np.integer)) or not 0<=label<len(label_map):
                    raise ValueError('Invalid original classification label index.')
                label=int(label);wnid=label_map[label];image=example['image'];blob=image.get('bytes')
                if not isinstance(blob,(bytes,bytearray)) or not blob: raise ValueError('Original source bytes are missing in the stream.')
                sha=hashlib.sha256(blob).hexdigest();key=image.get('path') or 'source-sha256:'+sha
                if not isinstance(key,str): raise ValueError('Original streaming image key is invalid.')
                target=(Path(output_split)/wnid/(f'{index:08d}-{sha[:24]}.npy')).as_posix()
                row=(split,index,key,sha,label,wnid,target)
                aggregate.update((json.dumps(row,separators=(',',':'))+'\n').encode())
                cached=conn.execute('SELECT image_key,source_sha,label_index,wnid,target,latent_sha,complete FROM rows WHERE split=? AND seq=?',(split,index)).fetchone()
                valid=False
                if cached and cached[:5]==row[2:] and cached[6]:
                    try:
                        value=np.load(cache_root/target,allow_pickle=False)
                        valid=value.shape==(4,32,32) and value.dtype==np.float32 and bool(np.isfinite(value).all()) and hashlib.sha256(value.tobytes()).hexdigest()==cached[5]
                    except (OSError,ValueError,EOFError): pass
                if cached and cached[:5]!=row[2:]:
                    raise ValueError('Ordered example/key/label changed within the pinned original revision; refuse to silently mix caches.')
                if not valid: todo.append((row,imagenet_stream_image(blob)))
                if len(todo)>=batch_size: process(todo);todo=[]
                counts[split]=index+1;classes[split][wnid]+=1
                if time.monotonic()-last_snapshot>=snapshot_seconds: process(todo);todo=[];checkpoint()
            process(todo);todo=[]
            if counts[split]!=expected[split]: raise ValueError('The complete original streaming split was not read: '+split)
        if not all(classes['train'].values()): raise ValueError('All 1000 original training classes are required.')
        val_per_class=expected['validation']//len(ids)
        if expected['validation']%len(ids) or any(n!=val_per_class for n in classes['validation'].values()):
            raise ValueError('Original validation must have exactly 50 images for each of the1000 classes.')
        if conn.execute('SELECT COUNT(*) FROM rows WHERE complete=1').fetchone()[0]!=sum(expected.values()):
            raise ValueError('The full original streaming latent manifest is incomplete.')
        actual={p.relative_to(cache_root).as_posix() for p in cache_root.rglob('*.npy')}
        required={row[0] for row in conn.execute('SELECT target FROM rows')}
        if actual!=required: raise ValueError('Streaming latent file inventory contains missing or extra rows.')
        ready=True;checkpoint()
        files={split:[str(cache_root/row[0]) for row in conn.execute('SELECT target FROM rows WHERE split=? ORDER BY seq',('validation' if split=='val' else 'train',))] for split in ['train','val']}
        print('Verified full original streamed ImageNet latents:',counts,'revision:',source['revision'],'regenerated:',regenerated)
        return {'cache_root':cache_root,'files_by_split':files,'counts':{'train':counts['train'],'val':counts['validation'],'test':0},
                'identity_sha256':identity_sha,'source_manifest_sha256':aggregate.hexdigest(),'regenerated_rows':regenerated,
                'verified_rows':sum(expected.values()),'cache_complete':True,'source_revision':source['revision']}
    except BaseException as error:
        ready=False;checkpoint(str(error));raise
    finally: conn.close()

In [ ]:
IMAGENET_SOURCE = os.environ.get('IMAGENET_SOURCE','local')  # local or hf_stream
if IMAGENET_SOURCE == 'local':
    ds = ImagesDS(path,'**/*.JPEG')
    imagenet_latent_targets,imagenet_source_metadata = full_imagenet_source(path,path_data,ds.files)
    dl = DataLoader(ds,batch_size=min(4,MAX_BATCH_SIZE),num_workers=fc.defaults.cpus)
    print('Required full local ImageNet source verified:',imagenet_source_metadata['counts'])
elif IMAGENET_SOURCE == 'hf_stream':
    # User approval and token are required; this call never accepts the access conditions.
    imagenet_stream = approved_imagenet_stream_source(path_data)
    imagenet_source_metadata = imagenet_stream['source']
    dl = [imagenet_stream_preview(imagenet_stream,min(4,MAX_BATCH_SIZE))]
    print('Original source selected with verified WNID mapping and pinned revision:',imagenet_source_metadata['revision'])
    print('Full source count/order checks occur during streaming; this preview is not completion evidence.')
else:
    raise ValueError('Use local or hf_stream; no alternate dataset route is supplied.')

In [ ]:
xb,yb = next(iter(dl))
xe = vae.encode(xb.cuda())
xs = xe.latent_dist.mean
xs.shape

In [ ]:
show_images(((xs[:16,:3])/4).sigmoid(), imsize=2)

In [ ]:
xd = to_cpu(vae.decode(xs))
show_images(xd['sample'][:16].clamp(0,1), imsize=2)

In [ ]:
# Per-row recovery verifies every original source/output before classification training.
imagenet_encoding = latent_encoder_settings(vae,{
    'rgb':'read_image/decode_image RGB /255','crop':'center square min(H,W)','resize':256,
    'label_sources':imagenet_source_metadata})
if IMAGENET_SOURCE == 'hf_stream':
    imagenet_latents = verified_streamed_imagenet_latents(
        imagenet_stream,vae,dest,Path('results')/'imagenet_latent_preparation',
        imagenet_encoding,batch_size=min(4,MAX_BATCH_SIZE))
    verified_latent_files = imagenet_latents['files_by_split']
    imagenet_source_counts = imagenet_latents['counts']
else:
    imagenet_latents = verified_full_latents(
        ds.files,path,lambda index:ds[index][0],vae,dest,
        Path('results')/'imagenet_latent_preparation',imagenet_encoding,
        mode='npy',targets=imagenet_latent_targets,batch_size=min(4,MAX_BATCH_SIZE))
    verified_latent_files = {split:[filename for filename,target in zip(imagenet_latents['files'],imagenet_latent_targets)
                                   if Path(target).parts[0]==split] for split in ['train','val']}
    imagenet_source_counts = imagenet_source_metadata['counts']
assert imagenet_latents['cache_complete']
assert imagenet_source_counts['train'] == 1281167 and imagenet_source_counts['val'] == 50000
assert len(verified_latent_files['train']) == 1281167 and len(verified_latent_files['val']) == 50000
dest = imagenet_latents['cache_root']

In [ ]:
class NumpyDS(ImagesDS):
    def __init__(self,path,spec):
        # This exact source-derived row list has already passed count/shape/hash checks.
        self.files=verified_latent_files[Path(path).name]
    def __getitem__(self,i):
        f=self.files[i]
        im=np.load(f,allow_pickle=False)
        return im,f

In [ ]:
bs = min(128, MAX_BATCH_SIZE)

In [ ]:
tds = NumpyDS(dest/'train', '**/*.npy')
vds = NumpyDS(dest/'val', '**/*.npy')

In [ ]:
tdl = DataLoader(tds, batch_size=bs, num_workers=0)
xb,yb = next(iter(tdl))
xb.mean((0,2,3)),xb.std((0,2,3))

In [ ]:
xmean,xstd = (tensor([ 5.37007,  2.65468,  0.44876, -2.39154]),
 tensor([3.99512, 4.44317, 3.21629, 3.10339]))

In [ ]:
class TfmDS:
    def __init__(self, ds, tfmx=fc.noop, tfmy=fc.noop): self.ds,self.tfmx,self.tfmy = ds,tfmx,tfmy
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        x,y = self.ds[i]
        return self.tfmx(x),self.tfmy(y)

In [ ]:
id2str = (path_data/'imagenet_lsvrc_2015_synsets.txt').read_text().splitlines()
str2id = {v:k for k,v in enumerate(id2str)}

In [ ]:
aug_tfms = nn.Sequential(T.Pad(2), T.RandomCrop(32), RandErase())
norm_tfm = T.Normalize(xmean, xstd)

In [ ]:
def tfmx(x, aug=False):
    x = norm_tfm(tensor(x))
    if aug: x = aug_tfms(x[None])[0]
    return x

def tfmy(y): return tensor(str2id[Path(y).parent.name])

tfm_tds = TfmDS(tds, partial(tfmx, aug=True), tfmy)
tfm_vds = TfmDS(vds, tfmx, tfmy)

In [ ]:
def denorm(x): return (x*xstd[:,None,None]+xmean[:,None,None])

In [ ]:
dls = DataLoaders(*get_dls(tfm_tds, tfm_vds, bs=bs, num_workers=2))

In [ ]:
all_synsets = [o.split('\t') for o in (path_data/'words.txt').read_text().splitlines()]
synsets = {k:v.split(',', maxsplit=1)[0] for k,v in all_synsets if k in id2str}

In [ ]:
xb,yb = next(iter(dls.train))
titles = [synsets[id2str[o]] for o in yb]
xb.mean(),xb.std()

In [ ]:
xd = to_cpu(vae.decode(denorm(xb[:9]).cuda()))
show_images(xd['sample'].clamp(0,1), imsize=4, titles=titles[:9])

In [ ]:
act_gr = partial(GeneralRelu, leak=0.1, sub=0.4)
iw = partial(init_weights, leaky=0.1)

opt_func = partial(optim.AdamW, eps=1e-5)
metrics = MetricsCB(accuracy=MulticlassAccuracy())
cbs = [DeviceCB(), metrics, ProgressCB(plot=True), MixedPrecision()]

In [ ]:
def conv(ni, nf, ks=3, stride=1, act=nn.ReLU, norm=None, bias=True):
    layers = []
    if norm: layers.append(norm(ni))
    if act : layers.append(act())
    layers.append(nn.Conv2d(ni, nf, stride=stride, kernel_size=ks, padding=ks//2, bias=bias))
    return nn.Sequential(*layers)

def _conv_block(ni, nf, stride, act=act_gr, norm=None, ks=3):
    return nn.Sequential(conv(ni, nf, stride=1     , act=act, norm=norm, ks=ks),
                         conv(nf, nf, stride=stride, act=act, norm=norm, ks=ks))

class ResBlock(nn.Module):
    def __init__(self, ni, nf, stride=1, ks=3, act=act_gr, norm=None):
        super().__init__()
        self.convs = _conv_block(ni, nf, stride, act=act, ks=ks, norm=norm)
        self.idconv = fc.noop if ni==nf else conv(ni, nf, ks=1, stride=1, act=None, norm=norm)
        self.pool = fc.noop if stride==1 else nn.AvgPool2d(2, ceil_mode=True)

    def forward(self, x): return self.convs(x) + self.idconv(self.pool(x))

def res_blocks(n_bk, ni, nf, stride=1, ks=3, act=act_gr, norm=None):
    return nn.Sequential(*[
        ResBlock(ni if i==0 else nf, nf, stride=stride if i==n_bk-1 else 1, ks=ks, act=act, norm=norm)
        for i in range(n_bk)])

def get_dropmodel(nfs, nbks, act=act_gr, norm=nn.BatchNorm2d, drop=0.2):
    layers = [nn.Conv2d(4, nfs[0], 5, padding=2)]
    layers += [res_blocks(nbks[i], nfs[i], nfs[i+1], act=act, norm=norm, stride=2)
               for i in range(len(nfs)-1)]
    layers += [act_gr(), norm(nfs[-1]), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(drop)]
    layers += [nn.Linear(nfs[-1], 1000, bias=False), nn.BatchNorm1d(1000)]
    return nn.Sequential(*layers).apply(iw)

In [ ]:
epochs = 40
lr = 1e-2
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched)]
model = get_dropmodel(nbks=(1,2,4,3), nfs=(32, 64, 128, 512, 1024), drop=0.1)
learn = Learner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=opt_func)

In [ ]:
learn.fit(epochs)

In [ ]:
torch.save(learn.model, 'models/imgnet-latents')

### fin -

Save your outputs

Run the following cell after finishing. Choose **File > Save** in Colab to preserve your code and displayed results. Files in `results/` and `models/` persist in Drive when enabled.

In [ ]:
finish_run(COLAB_CONTEXT)

## Full ImageNet inputs and recoverable latent preparation

This is the original 1000-class **classification** experiment over VAE mean latents, with 40 training epochs. It requires all 1,281,167 training and 50,000 validation images, plus the original 1000-synset-order and words files. The local route maps flat validation JPEGs into the correct latent class folders using official validation XML or `LOC_val_solution.csv`; it does not move raw images. An optional complete local test split is encoded but is not used by this classifier.

The train/validation float32(4,32,32) numeric latent payload is 21,809,840,128 bytes. NPY headers and filesystem allocation are additional: on a filesystem with 4 KiB blocks, these latent files occupy approximately 27,262,300,160 bytes before manifests/directories/models. All 100,000 optional unlabelled test images add 1,638,400,000 numeric payload bytes. Raw images and latents stay runtime-local. Compact row manifests/status and existing models/results persist to Drive. A manifest on a new runtime does not restore data; missing latents must be regenerated. Old unverified caches are preserved; verified caches use encoding/source namespaces.

Local preparation checks current source byte hashes on every attempt and verifies every latent's shape, dtype, finite values and SHA256 before training. Atomic writes/flushes precede progress commits. Only invalid or missing rows are encoded; a partial allocation or any existing NPY file cannot establish completeness. Original RGB/crop/resize, float32 latent means, provided normalization statistics, model and 40-epoch schedule are retained.

No exact course-author precomputed ImageNet latent URL was found in the inspected original notebook/repository, README, lesson 25 resources or official lesson topic. This bounded search does not establish that no such files exist elsewhere. The local route needs original filesystem data. The optional approved official streaming route below avoids retaining all raw images; its actual pinned-revision label features still need live verification with the learner's approved account. With roughly 10 GB free Drive, the full latent payload cannot be persisted there. No ImageNet source access, hosted encoding/training result or course completion is claimed by preparation.

[ImageNet access/counts](https://www.image-net.org/download.php), [official HF access conditions](https://huggingface.co/datasets/ILSVRC/imagenet-1k), [original experiment](https://github.com/fastai/course22p2/blob/df9323235bc395b5c2f58a3d08b83761947b9b93/nbs/31_imgnet_latents-widish.ipynb).
No terms are accepted or contact information shared by this preparation.


### Optional approved-access original ImageNet streaming

The default `IMAGENET_SOURCE=local` uses the original JPEG files. To avoid storing all raw ImageNet on a limited Colab disk, select `IMAGENET_SOURCE=hf_stream` in the environment before the source-selection cell. This route uses only the official `ILSVRC/imagenet-1k` repository. The learner must personally obtain approved access, provide `HF_TOKEN` through Colab Secrets/environment and set `IMAGENET_APPROVED_ACCESS=1`. The notebook does not accept terms, share contact details or save the token. `IMAGENET_HF_REVISION` may specify a commit; any requested revision is resolved and recorded as an immutable commit before iteration.

Supply the original `imagenet_lsvrc_2015_synsets.txt` and `words.txt` under `data/ILSVRC`. The source revision must expose a classification `ClassLabel` whose 1000 distinct names are the exact original WNIDs; the code checks the builder and both split features. Human display names or an unverifiable label order cause an explicit failure. No class order is inferred from the dataset-card display names. The original classifier label order remains the supplied synset-file order.

All 1,281,167 training and 50,000 validation examples are streamed without shuffling the source. A per-row ledger binds the revision, ordered image key/source bytes, numeric label/WNID, encoder and latent hash. Every original class and 50 validation images per class are required. Interrupted encoding reuses only rows that match this evidence and pass current shape/dtype/hash checks. A changed ordered source within the pinned revision is refused. A cache marker stays incomplete until all rows pass. Source replay may transfer the full dataset again after an interruption; the code regenerates only missing or invalid latents.

Raw images are decoded one microbatch at a time and discarded. Float32 latent files remain runtime-local. Numeric train+validation payload is **21,809,840,128 bytes**; NPY headers and 4 KiB filesystem allocation increase the latent files to approximately **27,262,300,160 bytes**, plus SQLite/directory/model overhead. The notebook checks free space and file entries. Compact manifests/status and existing model/checkpoint/results hooks use Drive. A runtime reset requires original-source replay and latent regeneration; the manifest alone is never treated as data. The optional 100,000 unlabelled test images are not needed for this classifier.

The public official HF card reports a **167 GB** repository including test; streaming avoids retaining that repository on disk, but does not remove remote transfer/encoding time or access requirements. This adapter was prepared and checked with small fixtures; **no approved-access ImageNet download, hosted encoding, 40-epoch training or course completion is claimed**. See the [official dataset card](https://huggingface.co/datasets/ILSVRC/imagenet-1k), [streaming documentation](https://huggingface.co/docs/datasets/stream) and [ImageNet download page](https://www.image-net.org/download.php).
